# Classical Machine Learning #

In [ ]:
from pathlib import Path
import time
import json

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, matthews_corrcoef, precision_recall_curve
import sys
from pathlib import Path

scripts_dir = (Path.cwd() / ".." / "scripts").resolve()
sys.path.append(str(scripts_dir))

from config import final_data_path

# Config
final_data_path = Path(final_data_path)
features_dir = final_data_path

layer_files = [
    "layer_a_bts",
    "layer_b_bts_meteostat",
    "layer_c_bts_gdelt",
    "layer_d_bts_meteostat_gdelt",
]
variants = ["", "_subsampled"]

target_col = "dep_del15"
non_feature_cols = ["flight_id", "year", target_col]

datasets = {}
for name in layer_files:
    for variant in variants:
        key = f"{name}{variant}"
        df = pd.read_parquet(features_dir / f"{key}_features.parquet")
        datasets[key] = df
        print(f"{key}: {df.shape}")


def get_train_test(key: str):
    """Split one dataset by year and drop non-feature columns from X."""
    df = datasets[key]
    train = df[df["year"] == 2024]
    test = df[df["year"] == 2025]

    feature_cols = [c for c in df.columns if c not in non_feature_cols]
    X_train, y_train = train[feature_cols], train[target_col]
    X_test, y_test = test[feature_cols], test[target_col]
    return X_train, y_train, X_test, y_test

layer_a_bts: (12028696, 28)
layer_a_bts_subsampled: (6966559, 28)
layer_b_bts_meteostat: (12028696, 32)
layer_b_bts_meteostat_subsampled: (6966559, 32)
layer_c_bts_gdelt: (12028696, 48)
layer_c_bts_gdelt_subsampled: (6966559, 48)
layer_d_bts_meteostat_gdelt: (12028696, 52)
layer_d_bts_meteostat_gdelt_subsampled: (6966559, 52)


In [ ]:
RF_PARAMS = {
    "n_estimators": 300,
    "max_depth": None,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "oob_score": True,
    "n_jobs": -1,
    "random_state": 42,
}

FULL_LAYERS = [
    "layer_a_bts",
    "layer_b_bts_meteostat",
    "layer_c_bts_gdelt",
    "layer_d_bts_meteostat_gdelt",
]

results_dir = final_data_path / "modeling_results" / "rf"
results_dir.mkdir(parents=True, exist_ok=True)


def tune_threshold(y_true, oob_proba):
    """Pick the threshold that maximizes F1 on OOB (in-sample, unbiased) predictions."""
    precision, recall, thresholds = precision_recall_curve(y_true, oob_proba)
    f1_scores = 2 * precision * recall / (precision + recall + 1e-12)
    best_idx = np.nanargmax(f1_scores[:-1])  # last precision/recall pair has no threshold
    return thresholds[best_idx]


run_results = {}

for key in FULL_LAYERS:
    print(f"\n{'=' * 60}\n{key}\n{'=' * 60}")

    X_train, y_train, X_test, y_test = get_train_test(key)
    print(f"Train: {X_train.shape}, Test: {X_test.shape}")

    model = RandomForestClassifier(**RF_PARAMS)

    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    print(f"Fit time: {elapsed:.1f}s | OOB score: {model.oob_score_:.4f}")

    # OOB probabilities -> tune threshold without touching 2025 or refitting
    oob_proba = model.oob_decision_function_[:, 1]
    threshold = tune_threshold(y_train, oob_proba)
    print(f"Tuned threshold (from OOB): {threshold:.4f}")

    # Real evaluation on 2025 holdout, frozen threshold
    test_proba = model.predict_proba(X_test)[:, 1]
    test_pred = (test_proba >= threshold).astype(int)

    metrics = {
        "f1": f1_score(y_test, test_pred),
        "auc_roc": roc_auc_score(y_test, test_proba),
        "auc_pr": average_precision_score(y_test, test_proba),
        "mcc": matthews_corrcoef(y_test, test_pred),
        "threshold": float(threshold),
        "oob_score": float(model.oob_score_),
        "fit_time_s": elapsed,
    }
    print(f"F1: {metrics['f1']:.4f} | AUC-ROC: {metrics['auc_roc']:.4f} | "
          f"AUC-PR: {metrics['auc_pr']:.4f} | MCC: {metrics['mcc']:.4f}")

    # Save probabilities (needed later for DeLong's test) and metrics
    np.savez(
        results_dir / f"{key}_probs.npz",
        y_test=y_test.to_numpy(),
        test_proba=test_proba,
    )
    with open(results_dir / f"{key}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    run_results[key] = metrics

print(f"\n{'=' * 60}\nSummary\n{'=' * 60}")
for key, m in run_results.items():
    print(f"{key:32s} F1={m['f1']:.4f} AUC-ROC={m['auc_roc']:.4f} "
          f"AUC-PR={m['auc_pr']:.4f} MCC={m['mcc']:.4f}")

Layer A (full): 6,062,137 train rows, 25 features

Fit time (50 trees): 79.7s
Estimated time at 300 trees: 478.2s (~8.0 min)
Estimated time at 500 trees: 797.0s (~13.3 min)
